In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1) 데이터 로드
df = pd.read_csv("train.csv")

# 2) 간단한 EDA
print("\n결측치:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\n타겟 비율:\n", df["Survived"].value_counts(normalize=True))

# 3) Name 컬럼 추가
use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Name"]
df = df[use_cols].copy()

y = df["Survived"]
X = df.drop(columns=["Survived"])

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# (A) 컬럼 구분
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked"]
text_col = "Name"

# (B) 수치형 파이프라인
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  # 트리는 스케일링 필수는 아니지만, 그대로 두어도 동작합니다.
])

# (C) 범주형 파이프라인
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# (D) 텍스트 파이프라인 --> 이건 이번에는 빼는걸로 (딱히 실속 X)
# text_pipeline = Pipeline([
#     ("vectorizer", CountVectorizer(min_df=2))
# ])

# (E) 전처리 통합
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
])

# (F) 전체 Pipeline 구성 (의사결정트리로 변경)
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42))
])

# 8) GridSearchCV (트리 하이퍼파라미터 튜닝 예시)
param_grid = {
    "clf__max_depth": [None, 3, 5, 7, 10],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 5]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# 테스트 평가
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


결측치:
 Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
dtype: int64

타겟 비율:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Best Params: {'clf__max_depth': 7, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 2}
Best CV Score: 0.8217374175120653

Test Accuracy: 0.7877094972067039

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.88      0.84       110
           1       0.77      0.64      0.70        69

    accuracy                           0.79       179
   macro avg       0.78      0.76      0.77       179
weighted avg       0.79      0.79      0.78       179



In [3]:
# 여기서도 확률값 뽑아보기
best_model.predict_proba(X_test)[0]

array([0.9375, 0.0625])

In [4]:
best_model.predict(X_test)[0]

np.int64(0)

# 나중에 굉장히 중요하고 요긴하게 사용할 기능 (feature importance)

In [5]:
importances = best_model.named_steps["clf"].feature_importances_


In [6]:
importances

array([0.16988689, 0.13579143, 0.00149658, 0.00105247, 0.16896151,
       0.4884004 , 0.        , 0.00288626, 0.        , 0.03152447])

In [7]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'Name'],
      dtype='object')

In [8]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Name
0,0,3,male,22.0,1,0,7.2500,S,"Braund, Mr. Owen Harris"
1,1,1,female,38.0,1,0,71.2833,C,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,1,3,female,26.0,0,0,7.9250,S,"Heikkinen, Miss. Laina"
3,1,1,female,35.0,1,0,53.1000,S,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,0,3,male,35.0,0,0,8.0500,S,"Allen, Mr. William Henry"
...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,"Montvila, Rev. Juozas"
887,1,1,female,19.0,0,0,30.0000,S,"Graham, Miss. Margaret Edith"
888,0,3,female,NaN,1,2,23.4500,S,"Johnston, Miss. Catherine Helen ""Carrie"""
889,1,1,male,26.0,0,0,30.0000,C,"Behr, Mr. Karl Howell"


In [14]:
fi

,feature,importance,group
0,Pclass,0.169887,Pclass
1,Age,0.135791,Age
2,SibSp,0.001497,SibSp
3,Parch,0.001052,Parch
4,Fare,0.168962,Fare
5,Sex_female,0.488400,Sex
6,Sex_male,0.000000,Sex
7,Embarked_C,0.002886,Embarked
8,Embarked_Q,0.000000,Embarked
9,Embarked_S,0.031524,Embarked


In [9]:
best_model = grid.best_estimator_
pre = best_model.named_steps["preprocess"]
clf = best_model.named_steps["clf"]

importances = clf.feature_importances_

# feature name 가져오기 (num + cat만)
num_features = num_cols

ohe = pre.named_transformers_["cat"].named_steps["onehot"] # OneHotEncoder 객체를 가져오는 코드
cat_features = list(ohe.get_feature_names_out(cat_cols)) # OneHotEncoder가 만든 컬럼 이름을 가져오는 코드

feature_names = list(num_features) + cat_features # 숫치형 컬럼 + (원-핫 인코딩된) 범주형 컬럼 합치기

# 각 컬럼의 이름과 feature importance 가지고 데이터프레임 형태 만들기
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances[:len(feature_names)]
})

# 컬럼 단위로 합산
#Sex_male
#Sex_female
#이런 것들을 "Sex" 그룹으로 묶이게끔
fi["group"] = fi["feature"].apply(
    lambda x: x.split("_")[0] if "_" in x else x
)

# group 기준으로 묶고 컬럼 하나당 중요도 하나씩 나오게끔
column_importance = (
    fi.groupby("group")["importance"]
      .sum()
      .sort_values(ascending=False)
)

print("\n컬럼별 중요도:")
print(column_importance)


컬럼별 중요도:
group
Sex         0.488400
Pclass      0.169887
Fare        0.168962
Age         0.135791
Embarked    0.034411
SibSp       0.001497
Parch       0.001052
Name: importance, dtype: float64
